# Faruq-v3 DSRDet SSCB/MSDA breadth screening

Seed-42 validation-only screen for three arms: **M0** = MSDA calibration without semantic auxiliary supervision, **S0** = semantic auxiliary supervision + MSDA, **S1** = full calibrated SSCB transfer. This is a YOLO26 transfer of DSRDet SSCB/MSDA, not a literal DSRDet reproduction. Localization/TAL stay native. Test is never restored/opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/dsr-sscb-msda-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone=['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(3):
    r=subprocess.run(clone)
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('Git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(),'Aktifkan GPU.'
PROJECT_ROOT=resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json',
))
ARCHIVE=require_project_artifact(PROJECT_ROOT,'bundles/faruq-development-v3-grouped.tar')
D0=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
D0FT=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json')
ACMC1=require_project_artifact(PROJECT_ROOT,'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT=Path('/content/faruq-development-v3-grouped')
GROUPED=DATA_ROOT/'faruq_grouped_summary.json'
if not GROUPED.is_file():
    with tarfile.open(ARCHIVE,'r') as a: a.extractall('/content',filter='data')
assert GROUPED.is_file(); assert not (DATA_ROOT/'test').exists()
OUTPUT=PROJECT_ROOT/'experiments/faruq-v3-dsr-sscb-screening-v1'
print('GPU:',torch.cuda.get_device_name(0)); print('OUTPUT:',OUTPUT)


In [ ]:
cmd=[sys.executable,'-m','pytest','-q','tests/test_dsr_sscb.py']
print('STATIC CHECK:',' '.join(cmd)); subprocess.run(cmd,cwd=REPO,check=True)


In [ ]:
cmd=[
    sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_dsr_sscb_screening',
    '--data-root',str(DATA_ROOT),'--grouped-summary',str(GROUPED),
    '--d0-checkpoint',str(D0),'--d0ft-report',str(D0FT),'--acmc1-report',str(ACMC1),
    '--output-root',str(OUTPUT),'--seed','42','--device','0','--authorize-training',
]
print('MENJALANKAN:',' '.join(cmd),flush=True)
p=subprocess.run(cmd,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT); print(p.stdout)
if p.returncode: raise RuntimeError(f'DSR SSCB screening gagal: {p.returncode}')


In [ ]:
import json,pandas as pd
from IPython.display import display
SUMMARY=OUTPUT/'val_reports/dsr_sscb_seed42_screening.json'
r=json.loads(SUMMARY.read_text(encoding='utf-8'))
assert r['evaluation_split']=='val'; assert r['test_opened'] is False and r['test_images_accessed'] is False
rows=[{'model':name,**metrics} for name,metrics in r['results'].items()]
display(pd.DataFrame(rows).style.format({k:'{:.2%}' for k in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')}))
for arm in ('M0','S0','S1'):
    print(arm,r['decisions'][arm],r['deltas'][arm]['vs_D0FT'])
print('ATTRIBUTION:',r['attribution']); print('SUMMARY:',SUMMARY)
